# Getting Started: Your First Agent

In this notebook, we'll walk through creating your first Claude agent using the SDK. We'll start simple and progressively add complexity.

## Setup

Before we start, let's set up the notebook environment to run async code properly:

In [4]:
# Setup for running async code in Jupyter
import nest_asyncio
nest_asyncio.apply()

# Load optional environment variables from a .env file (if present).
# This course can use your Claude subscription, so no ANTHROPIC_API_KEY is needed --
# load_dotenv() is here only for optional keys (e.g. TAVILY_API_KEY), or an
# ANTHROPIC_API_KEY if you choose per-token API billing instead.
from dotenv import load_dotenv
load_dotenv()

print("✓ Notebook environment configured")

✓ Notebook environment configured


## Language Support

### SDK Available in Two Languages

The Claude Agent SDK is available in both **Python** and **TypeScript** with full feature parity.

```
Python SDK                        TypeScript SDK

pip install                       npm install
claude-agent-sdk                  @anthropic-ai/claude-agent-sdk
```

### This Course Uses Python

All examples, exercises, and hands-on labs will use Python. The concepts translate directly to TypeScript if you prefer.

| Resource | Link |
|----------|------|
| Python SDK Docs | [platform.claude.com/docs/en/agent-sdk/python](https://platform.claude.com/docs/en/agent-sdk/python) |
| TypeScript SDK Docs | [platform.claude.com/docs/en/agent-sdk/typescript](https://platform.claude.com/docs/en/agent-sdk/typescript) |

## Installation

### Prerequisites

- Python 3.10 or higher
- The **Claude Code CLI**, installed and logged in — the Python SDK calls it under the hood (it is *not* bundled with `claude-agent-sdk`)
- A **Claude Pro/Max subscription** (this course's default) — or an Anthropic API key if you prefer per-token billing

### Install the SDK

If you're using this repo with `uv`, dependencies are already installed. Otherwise, uncomment and run the cell below:

In [5]:
# Uncomment to install the Python SDK
# !pip install claude-agent-sdk

# The SDK also needs the Claude Code CLI, installed and logged in separately:
#   curl -fsSL https://claude.ai/install.sh | bash   # macOS / Linux
#   claude                                            # then log in via the browser

## Authentication

This course runs on your **Claude Pro/Max subscription** — no API key or per-token billing required.

Once the Claude Code CLI is installed (see above), log in with your Claude account:

```bash
claude        # opens a browser to log in (use /login if not prompted)
```

Inside `claude`, run `/status` to confirm you're authenticated via your subscription.

> ⚠️ **Precedence gotcha:** if `ANTHROPIC_API_KEY` is set in your environment, the SDK uses it (per-token API billing) and **ignores your subscription**. For subscription use, keep `ANTHROPIC_API_KEY` unset — don't put it in `.env`.

See the [Claude Code quickstart](https://code.claude.com/docs/en/quickstart) for Windows and other install options.

### Optional: use an API key instead

Prefer per-token API billing? You don't need to change any code — just make the key available in the environment and the SDK picks it up automatically:

```bash
# Option A - shell environment variable
export ANTHROPIC_API_KEY="sk-ant-..."
```

```bash
# Option B - add to a .env file in the project (the setup cell above calls load_dotenv())
# ANTHROPIC_API_KEY=sk-ant-...
```

```python
# (Optional) confirm which auth the SDK will use:
# import os
# print("API key billing" if os.environ.get("ANTHROPIC_API_KEY") else "Claude subscription")
```

## Your First Agent

### Minimal Example

Let's create the simplest possible agent. This agent will:
1. Import the `query` function - the main entry point
2. Call `query()` with a prompt
3. Stream messages back as the agent works
4. Automatically use appropriate tools (Glob, Bash, etc.)

**That's it** - the agent loop, tools, and context management are all handled for you.

Run the cell below to see it in action:

In [6]:
import asyncio
from claude_agent_sdk import query

async def main():
    async for message in query(prompt="What files are in this directory?"):
        print(message)

await main()

SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': '/Users/sajal/projects/teaching/oreilly/claude_agent_sdk_course/claude_agent_sdk_course_code/code/examples/module_1', 'session_id': '831648b2-b0dd-4031-b098-ee7aff0c2ac7', 'tools': ['Task', 'AskUserQuestion', 'Bash', 'CronCreate', 'CronDelete', 'CronList', 'DesignSync', 'Edit', 'EnterPlanMode', 'EnterWorktree', 'ExitPlanMode', 'ExitWorktree', 'Monitor', 'NotebookEdit', 'PushNotification', 'Read', 'RemoteTrigger', 'ScheduleWakeup', 'Skill', 'TaskCreate', 'TaskGet', 'TaskList', 'TaskOutput', 'TaskStop', 'TaskUpdate', 'ToolSearch', 'WebFetch', 'WebSearch', 'Workflow', 'Write'], 'mcp_servers': [{'name': 'claude.ai Gmail', 'status': 'pending'}], 'model': 'claude-opus-4-8[1m]', 'permissionMode': 'default', 'slash_commands': ['slide-reviewer', 'youtube-watcher', 'commit', 'pr', 'deep-research', 'review-loop:cancel-review', 'review-loop:review-loop', 'document-skills:algorithmic-art', 'document-skills:brand-guideli

### Making Output More Readable

The raw message output above shows all the internal details. Let's create a helper function to print messages in a cleaner, more readable format:

In [7]:
import json

def print_message(message):
    """Pretty print agent messages."""
    msg_type = type(message).__name__
    
    if msg_type == "SystemMessage":
        # Skip system messages for cleaner output
        pass
    
    elif msg_type == "AssistantMessage":
        # Print assistant thinking and tool use
        if hasattr(message, 'content'):
            for block in message.content:
                block_type = type(block).__name__
                if block_type == "TextBlock":
                    print(f"🤖 Assistant: {block.text}")
                elif block_type == "ToolUseBlock":
                    print(f"🔧 Tool: {block.name}")
                    if hasattr(block, 'input'):
                        # Show description first if available
                        if 'description' in block.input:
                            print(f"   → {block.input['description']}")
                        # Show other arguments (excluding description)
                        args = {k: v for k, v in block.input.items() if k != 'description'}
                        if args:
                            print(f"   Arguments: {json.dumps(args, indent=6)}")
    
    elif msg_type == "UserMessage":
        # Print tool results
        if hasattr(message, 'content'):
            for block in message.content:
                block_type = type(block).__name__
                if block_type == "ToolResultBlock":
                    if block.is_error:
                        print(f"❌ Tool Error: {block.content}")
                    else:
                        # Show first 500 chars of result
                        content = str(block.content)
                        if len(content) > 500:
                            content = content[:500] + "..."
                        print(f"📤 Tool Result: {content}")
    
    elif msg_type == "ResultMessage":
        # Only show cost and timing metadata (result text is redundant with last AssistantMessage)
        if hasattr(message, 'total_cost_usd') and hasattr(message, 'duration_ms'):
            print(f"\n💰 Cost: ${message.total_cost_usd:.4f} | ⏱️ Time: {message.duration_ms/1000:.1f}s")

# Test it with a simple query
print("Testing the helper function:\n")
async def test_helper():
    async for message in query(prompt="What files are in this directory?"):
        print_message(message)

await test_helper()

Testing the helper function:

🤖 Assistant: I'll list the files in the current directory.
🔧 Tool: Bash
   → List files in current directory
   Arguments: {
      "command": "ls -la"
}
📤 Tool Result: drwxr-xr-x@   - sajal 24 Feb 13:04 .ipynb_checkpoints
.rw-r--r--@ 47k sajal 17 Jun 16:36 01_getting_started.ipynb
🤖 Assistant: Here's what's in the current directory:

- **`.ipynb_checkpoints/`** — directory (Jupyter auto-save checkpoints)
- **`01_getting_started.ipynb`** — Jupyter notebook (47k)

💰 Cost: $0.0544 | ⏱️ Time: 10.2s


Much better! The helper function filters out verbose system details and shows:
- 🤖 Assistant thinking
- 🔧 Tool usage with descriptions and arguments
- 📤 Tool results (truncated to 500 chars for readability)
- 💰 Cost and timing metadata

You can use `print_message()` in all the examples below for cleaner output.

## Two Input Modes

### Single Message vs Streaming Input

The SDK supports two distinct ways to interact with agents:

| Mode | Description | Best For |
|------|-------------|----------|
| **Single Message** | One-shot queries | Batch jobs, CI/CD, serverless |
| **Streaming Input** | Persistent interactive session | Interactive apps, multi-turn workflows |

### Quick Comparison

```
Single Message Input          Streaming Input Mode (Recommended)
─────────────────────         ─────────────────────────────────
One-shot tasks           →    Multi-turn conversations
Stateless environments   →    Stateful applications
No images                →    Image attachments supported
Simple automation        →    Interruption & queue handling
```

The example above uses **single message input** - the simplest way to get started.

## Single Message Input

### Continuing Conversations

Chain queries using session management. Try running these cells in sequence:

In [8]:
from claude_agent_sdk import ClaudeAgentOptions

# First query - analyze a file
async def first_query():
    async for msg in query(prompt="List the python notebook files in this directory"):
        print_message(msg)

await first_query()

🔧 Tool: Bash
   → Find Jupyter notebook files
   Arguments: {
      "command": "find /Users/sajal/projects/teaching/oreilly/claude_agent_sdk_course -name \"*.ipynb\" -not -path \"*/.venv/*\" -not -path \"*/.ipynb_checkpoints/*\""
}
📤 Tool Result: /Users/sajal/projects/teaching/oreilly/claude_agent_sdk_course/claude_agent_sdk_course_code/code/examples/module_3/01_custom_tools.ipynb
/Users/sajal/projects/teaching/oreilly/claude_agent_sdk_course/claude_agent_sdk_course_code/code/examples/module_3/02_multi_agent_systems.ipynb
/Users/sajal/projects/teaching/oreilly/claude_agent_sdk_course/claude_agent_sdk_course_code/code/examples/module_2/01_session_management.ipynb
/Users/sajal/projects/teaching/oreilly/claude_agent_sdk_course/claude_agent_...
🤖 Assistant: Here are the Jupyter notebook files in the repository (excluding `.venv` and checkpoints):

**Example notebooks** (`code/examples/`):
- `module_1/01_getting_started.ipynb`
- `module_2/01_session_management.ipynb`
- `module_2/02_hooks.ip

In [9]:
# Continue the conversation
async def continue_query():
    async for msg in query(
        prompt="what's in the getting started notebook",
        options=ClaudeAgentOptions(continue_conversation=True)
    ):
        print_message(msg)

await continue_query()

🔧 Tool: Read
   Arguments: {
      "file_path": "/Users/sajal/projects/teaching/oreilly/claude_agent_sdk_course/claude_agent_sdk_course_code/code/examples/module_1/01_getting_started.ipynb"
}
📤 Tool Result: [{'text': '<cell id="cell-0"><cell_type>markdown</cell_type># Getting Started: Your First Agent\n\nIn this notebook, we\'ll walk through creating your first Claude agent using the SDK. We\'ll start simple and progressively add complexity.</cell id="cell-0">\n<cell id="cell-1"><cell_type>markdown</cell_type>## Setup\n\nBefore we start, let\'s set up the notebook environment to run async code properly:</cell id="cell-1">\n<cell id="cell-2"># Setup for running async code in Jupyter\nimport nest_asy...
🤖 Assistant: The `01_getting_started.ipynb` notebook (in `code/examples/module_1/`) is the course's introduction to building a first agent with the Claude Agent SDK. Here's what it covers:

## Setup & Installation
- Configuring Jupyter for async with `nest_asyncio` + optional `load_dotenv

### Limitations

Single message input does **not** support:
- Image attachments in messages
- Dynamic message queueing
- Real-time interruption
- Hook integration

## Streaming Input Mode

### For Interactive Applications

Streaming input mode creates a **persistent interactive session** where you can:
- Send multiple messages over time
- Maintain conversation context across turns
- Queue messages dynamically
- Handle interruptions

This is the **recommended** approach for building interactive applications.

### How It Works

Use `ClaudeSDKClient` with an async generator that yields messages. The agent maintains state throughout the session and can reference previous messages.

In [10]:
from claude_agent_sdk import ClaudeSDKClient

async def interactive_chat():
    """
    Interactive chat session with the agent.
    The session persists across all your messages, maintaining context.
    """
    
    print("🤖 Interactive Chat Session")
    print("=" * 60)
    print("Chat with the agent about this project!")
    print("The agent will remember context across your messages.")
    print("Type 'exit' to end the session.")
    print("=" * 60)
    print()
    
    # Create a persistent session
    # NOTE: Using bypassPermissions for notebook convenience - auto-approves all tools
    # In production, use permission_mode="default" with canUseTool callback for user approval
    options = ClaudeAgentOptions(
        system_prompt={"type": "preset", "preset": "claude_code"},
        allowed_tools=["Read", "Bash", "Grep", "Glob"],
        permission_mode="bypassPermissions",  # ⚠️ Auto-approves all tool uses
        max_turns=20
    )
    
    async with ClaudeSDKClient(options) as client:
        while True:
            # Get user input
            user_input = input("👤 You: ").strip()
            
            if not user_input:
                continue
                
            if user_input.lower() in ['exit', 'quit', 'bye']:
                print("\n👋 Chat session ended!")
                break
            
            print()  # Blank line before agent response
            
            # Create a message generator for this turn
            async def send_message():
                yield {
                    "type": "user",
                    "message": {
                        "role": "user",
                        "content": user_input
                    }
                }
            
            # Send the message and get response
            try:
                await client.query(send_message())
                
                # Display the agent's response
                async for message in client.receive_response():
                    print_message(message)
                    # Stop after we get the final result
                    if type(message).__name__ == "ResultMessage":
                        break
                
                print()  # Blank line before next prompt
                
            except Exception as e:
                print(f"❌ Error: {e}\n")
                break

# Start the chat!
# Try asking questions like:
#   "What files are in this directory?"
#   "Tell me about the first notebook"
#   "How many cells does it have?"
await interactive_chat()

🤖 Interactive Chat Session
Chat with the agent about this project!
The agent will remember context across your messages.
Type 'exit' to end the session.



👤 You:  tell me about the first notebook



🔧 Tool: Read
   Arguments: {
      "file_path": "/Users/sajal/projects/teaching/oreilly/claude_agent_sdk_course/claude_agent_sdk_course_code/code/examples/module_1/01_getting_started.ipynb"
}
📤 Tool Result: [{'text': '<cell id="cell-0"><cell_type>markdown</cell_type># Getting Started: Your First Agent\n\nIn this notebook, we\'ll walk through creating your first Claude agent using the SDK. We\'ll start simple and progressively add complexity.</cell id="cell-0">\n<cell id="cell-1"><cell_type>markdown</cell_type>## Setup\n\nBefore we start, let\'s set up the notebook environment to run async code properly:</cell id="cell-1">\n<cell id="cell-2"># Setup for running async code in Jupyter\nimport nest_asy...
🤖 Assistant: `01_getting_started.ipynb` (in `code/examples/module_1/`) is the course's opening notebook — the foundation everything else builds on. It introduces the Claude Agent SDK from zero. Here's the walkthrough:

## Setup & Installation
- Configures Jupyter for async with `nest_asy

👤 You:  how many cells does it have?



🤖 Assistant: The notebook has **27 cells** (cell-0 through cell-26).

Breakdown:
- **16 markdown cells** — explanations, tables, and the section headers
- **11 code cells** — setup, the `print_message` helper, the `query()`/`continue_conversation` examples, the interactive `ClaudeSDKClient` chat loop, and the tool/system-prompt configuration examples

💰 Cost: $0.3718 | ⏱️ Time: 7.5s



👤 You:  exit



👋 Chat session ended!


### What Streaming Input Enables

In the example above, notice how:
- The session persisted across **3 separate messages**
- The agent remembered context (could reference "that notebook" from the previous question)
- Messages were queued and processed sequentially
- We used `ClaudeSDKClient` with `message_generator()` to control the flow

| Capability | Description |
|------------|-------------|
| **Image Uploads** | Attach images for visual analysis (not shown but supported) |
| **Queued Messages** | Send multiple messages, process sequentially |
| **Context Persistence** | Agent remembers previous conversation turns |
| **Interruption** | Cancel or redirect mid-conversation |
| **Hooks Support** | Lifecycle hooks for custom behavior |

**Use streaming input mode** when building interactive applications, chatbots, or any multi-turn workflows.

## Configuring Agent Behavior

### Adding Options

You can control what tools the agent can use and how it behaves.

In [13]:
async def configured_agent():
    # NOTE: allowed_tools is a *permission allowlist* (which tool calls are
    # auto-approved), NOT an availability filter -- it does not remove built-in
    # tools. To block tools, use disallowed_tools (blacklist) as shown here.
    async for message in query(
        prompt="create a hello world python script",
        options=ClaudeAgentOptions(
            # Block write tools
            disallowed_tools=["Bash", "Edit", "Write", "NotebookEdit", "Agent", "Monitor"],
            permission_mode="bypassPermissions"
        )
    ):
        print_message(message)

await configured_agent()

🔧 Tool: ToolSearch
   Arguments: {
      "query": "select:Write",
      "max_results": 1
}
📤 Tool Result: No matching deferred tools found
🔧 Tool: ToolSearch
   Arguments: {
      "query": "write file create edit bash",
      "max_results": 5
}
📤 Tool Result: [{'type': 'tool_reference', 'tool_name': 'TaskCreate'}, {'type': 'tool_reference', 'tool_name': 'mcp__claude_ai_Gmail__create_draft'}, {'type': 'tool_reference', 'tool_name': 'mcp__claude_ai_Gmail__create_label'}, {'type': 'tool_reference', 'tool_name': 'CronCreate'}, {'type': 'tool_reference', 'tool_name': 'DesignSync'}]
🔧 Tool: ToolSearch
   Arguments: {
      "query": "select:Write,Edit,Bash",
      "max_results": 5
}
📤 Tool Result: No matching deferred tools found
🔧 Tool: ToolSearch
   Arguments: {
      "query": "execute shell command run script terminal",
      "max_results": 8
}
📤 Tool Result: [{'type': 'tool_reference', 'tool_name': 'PushNotification'}, {'type': 'tool_reference', 'tool_name': 'CronCreate'}, {'type': 'tool_

### Key Options

| Option | Purpose | Example |
|--------|---------|------|
| `disallowed_tools` | Block specific tools (blacklist) | `["Bash", "Write", "Edit"]` |
| `allowed_tools` | Permission allowlist — auto-approves these tool calls (not an availability filter) | `["Read", "Grep"]` |
| `permission_mode` | Control approval requirements | `"bypassPermissions"` |
| `system_prompt` | Customize agent instructions | `{"type": "preset", "preset": "claude_code"}` |
| `resume` | Continue previous session | `"session-id-123"` |

**Note:** `allowed_tools` is a *permission allowlist* — it controls which tool calls are auto-approved without prompting, not which tools exist. It does **not** remove built-in tools (see [issue #361](https://github.com/anthropics/claude-agent-sdk-python/issues/361)). To make a tool unavailable, use `disallowed_tools` (blacklist).

## System Prompt Configuration

### Important Gotcha

The SDK uses a **minimal system prompt by default** (just tool instructions).

### Get Claude Code Behavior

To get the full Claude Code CLI experience, explicitly configure it:

In [16]:
async def claude_code_style():
    options = ClaudeAgentOptions(
        system_prompt={"type": "preset", "preset": "claude_code"},  
        allowed_tools=["Read", "Glob", "Grep", "Bash"]
    )
    
    async for message in query(
        prompt="Tell me about the structure of this project",
        options=options
    ):
        print_message(message)

# await claude_code_style()

### What the `claude_code` preset Adds

Passing `system_prompt={"type": "preset", "preset": "claude_code"}` loads Claude Code's full system prompt, which adds:

- Code style and formatting guidelines
- Response tone and verbosity settings
- Security and safety instructions
- Context about the working directory
- Best practices for tool usage